<a href="https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes


The action playbook converts model or performance signals into a ranked queue for human review.

Higher-priority items are surfaced first based on their measured score and supporting performance signals. Each item receives a reason code so that the recommended action is explainable rather than being based on a score alone.

The queue is intended for prioritization and decision-support. A high-ranked item does not mean that a content change should be made automatically. Human review is required before any refresh, rewrite, consolidation, or other content action.

In [3]:
##ML-10 — Prepare W04 baseline output

import os
import duckdb
import pandas as pd
from google.colab import userdata

# ---------------------------------------------------------
# 1. Create DuckDB connection
# ---------------------------------------------------------

con = duckdb.connect()

print("DuckDB connection created.")

# ---------------------------------------------------------
# 2. Configure Hugging Face
# ---------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

try:
    con.execute("DROP SECRET hf_token")
except:
    pass

con.execute(
    """
    CREATE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN ?
    )
    """,
    [HF_TOKEN]
)

print("HF_TOKEN configured successfully.")

# ---------------------------------------------------------
# 3. Load March 2026 data
# ---------------------------------------------------------

march_df = con.sql(
    """
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

print("March content-client rows:", len(march_df))

# ---------------------------------------------------------
# 4. W04 baseline scoring
# ---------------------------------------------------------

score_df = march_df.copy()

score_df["click_score"] = 0

score_df.loc[
    score_df["gsc_clicks"] <= 2,
    "click_score"
] = 2

score_df.loc[
    (score_df["gsc_clicks"] > 2) &
    (score_df["gsc_clicks"] <= 10),
    "click_score"
] = 1

score_df["position_score"] = 0

score_df.loc[
    score_df["gsc_avg_position"] > 5,
    "position_score"
] = 1

score_df.loc[
    score_df["gsc_avg_position"] > 10,
    "position_score"
] = 2

score_df.loc[
    score_df["gsc_avg_position"] > 20,
    "position_score"
] = 3

score_df["score"] = (
    score_df["click_score"] +
    score_df["position_score"]
)

score_df["reason_code"] = "LOW_SEARCH_PERFORMANCE"
score_df["action"] = "REVIEW_REFRESH"

# ---------------------------------------------------------
# 5. Rank
# ---------------------------------------------------------

score_df = score_df.sort_values(
    by=[
        "score",
        "gsc_clicks",
        "gsc_avg_position"
    ],
    ascending=[
        False,
        True,
        False
    ]
).reset_index(drop=True)

score_df["rank"] = range(
    1,
    len(score_df) + 1
)

# ---------------------------------------------------------
# 6. Final baseline queue
# ---------------------------------------------------------

baseline_queue = score_df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action"
    ]
].copy()

# ---------------------------------------------------------
# 7. Save in CURRENT runtime
# ---------------------------------------------------------

os.makedirs(
    "/content/work/outputs",
    exist_ok=True
)

baseline_path = "/content/work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(
    baseline_path,
    index=False
)

# ---------------------------------------------------------
# 8. Final verification
# ---------------------------------------------------------

print("\n========================================")
print("W04 BASELINE OUTPUT CREATED")
print("========================================")

print("Rows:", len(baseline_queue))
print("Columns:", list(baseline_queue.columns))
print("Duplicate rows:", baseline_queue.duplicated().sum())
print("File exists:", os.path.exists(baseline_path))
print("Output:", baseline_path)

print("\nScore distribution:")
display(
    baseline_queue["score"]
    .value_counts()
    .sort_index()
)

print("\nTop 10:")
display(
    baseline_queue.head(10)
)

DuckDB connection created.
HF_TOKEN configured successfully.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March content-client rows: 176738

W04 BASELINE OUTPUT CREATED
Rows: 176738
Columns: ['rank', 'client_hash_id', 'content_hash_id', 'score', 'reason_code', 'action']
Duplicate rows: 0
File exists: True
Output: /content/work/outputs/baseline_action_score.csv

Score distribution:


,count
score,
0,7795
1,12391
2,38691
3,48544
4,29160
5,40157



Top 10:


,rank,client_hash_id,content_hash_id,score,reason_code,action
0,1,client_08a6a72ff48e62c0,content_9e8c3b83214c180d,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
1,2,client_3ffa76342f366962,content_06589faf15cc8488,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
2,3,client_3ffa76342f366962,content_36cc2bda86ee726a,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
3,4,client_08a6a72ff48e62c0,content_11187e07e5ee9f43,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
4,5,client_3ffa76342f366962,content_efce4eda2b012964,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
5,6,client_3ffa76342f366962,content_0cec599cfeab8b7f,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
6,7,client_23a62021009f63c4,content_61b375eafb1d4c27,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
7,8,client_f623b01661d4bfe4,content_fc468c5940d16ea3,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
8,9,client_3ffa76342f366962,content_3758dd311e8033f7,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
9,10,client_3ffa76342f366962,content_d1b44ca865290810,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH


In [4]:
# ML-10 — Section 1: Ranked actions + reason codes

import os
import pandas as pd

baseline_path = "work/outputs/baseline_action_score.csv"

if not os.path.exists(baseline_path):
    raise FileNotFoundError(
        "baseline_action_score.csv was not found in work/outputs/. "
        "Make sure the Week-4 baseline output is available before running this cell."
    )

action_queue = pd.read_csv(baseline_path)

required_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action"
]

missing_columns = [
    col
    for col in required_columns
    if col not in action_queue.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

action_queue = action_queue.sort_values(
    by=["score", "rank"],
    ascending=[False, True]
).reset_index(drop=True)

action_queue["rank"] = range(1, len(action_queue) + 1)

ranked_action_queue = action_queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action"
    ]
].copy()

print("Ranked action queue created successfully.")
print("Number of items:", len(ranked_action_queue))

print("\nReason-code distribution:")
display(
    ranked_action_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="count")
)

print("\nAction distribution:")
display(
    ranked_action_queue["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="count")
)

print("\nTop 10 ranked actions:")
display(ranked_action_queue.head(10))

Ranked action queue created successfully.
Number of items: 176738

Reason-code distribution:


,reason_code,count
0,LOW_SEARCH_PERFORMANCE,176738



Action distribution:


,action,count
0,REVIEW_REFRESH,176738



Top 10 ranked actions:


,rank,client_hash_id,content_hash_id,score,reason_code,action
0,1,client_08a6a72ff48e62c0,content_9e8c3b83214c180d,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
1,2,client_3ffa76342f366962,content_06589faf15cc8488,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
2,3,client_3ffa76342f366962,content_36cc2bda86ee726a,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
3,4,client_08a6a72ff48e62c0,content_11187e07e5ee9f43,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
4,5,client_3ffa76342f366962,content_efce4eda2b012964,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
5,6,client_3ffa76342f366962,content_0cec599cfeab8b7f,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
6,7,client_23a62021009f63c4,content_61b375eafb1d4c27,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
7,8,client_f623b01661d4bfe4,content_fc468c5940d16ea3,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
8,9,client_3ffa76342f366962,content_3758dd311e8033f7,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
9,10,client_3ffa76342f366962,content_d1b44ca865290810,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH


## 2. Intended use and limits

### Intended use

The action playbook is designed for human review and prioritization of content that may need attention. It uses observed performance signals and the measured baseline score to rank content items and provide reason codes for the recommended action.

The playbook can help SEO or content teams decide which items to review first. It is decision-support only and does not automatically change, delete, rewrite, or consolidate content.

### Limits

The recommendations are based on observed historical data and measured performance signals. They are directional and should not be treated as causal conclusions or guaranteed future outcomes.

A high score does not prove that a content item will decline or that a specific action will improve performance. Human review is required before taking action, and the recommendations may become less reliable if the underlying data, traffic patterns, or content environment changes.

In [5]:
# ML-10 — Section 2: Intended use and limits

print("Intended-use check")
print("------------------")
print("Purpose: prioritize content items for human review.")
print("Output type: ranked decision-support queue.")
print("Automation level: no automatic content changes.")
print("Evidence type: observed historical performance signals.")
print("Claim type: directional, not causal.")
print("Human review required: YES")

Intended-use check
------------------
Purpose: prioritize content items for human review.
Output type: ranked decision-support queue.
Automation level: no automatic content changes.
Evidence type: observed historical performance signals.
Claim type: directional, not causal.
Human review required: YES


## 3. Human review + the no-go list

### Action Playbook Validation

The action playbook was validated for required fields, ranking order, score range, reason codes, recommended actions, and duplicate client-content pairs.

All required columns are present and the ranking sequence is valid. Scores remain within the defined 0–5 range, reason codes and actions follow the baseline playbook definitions, and no duplicate client-content pairs are present.

**Validation result: PASS**

In [6]:
# ML-10 — Section 3: Action playbook checks

print("Action playbook validation")
print("--------------------------")

# Check required columns
required_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action"
]

missing_columns = [
    col
    for col in required_columns
    if col not in ranked_action_queue.columns
]

print("Required columns present:", len(missing_columns) == 0)

# Check rank sequence
expected_ranks = range(1, len(ranked_action_queue) + 1)

rank_check = (
    ranked_action_queue["rank"].tolist()
    == list(expected_ranks)
)

print("Rank sequence valid:", rank_check)

# Check score range
score_check = ranked_action_queue["score"].between(0, 5).all()

print("Score range valid:", score_check)

# Check reason codes
reason_check = (
    ranked_action_queue["reason_code"]
    .eq("LOW_SEARCH_PERFORMANCE")
    .all()
)

print("Reason codes valid:", reason_check)

# Check actions
action_check = (
    ranked_action_queue["action"]
    .eq("REVIEW_REFRESH")
    .all()
)

print("Actions valid:", action_check)

# Check duplicate content-client pairs
duplicate_check = (
    ranked_action_queue[
        ["client_hash_id", "content_hash_id"]
    ].duplicated().sum()
)

print("Duplicate client-content pairs:", duplicate_check)

# Overall result
section_3_pass = (
    len(missing_columns) == 0
    and rank_check
    and score_check
    and reason_check
    and action_check
    and duplicate_check == 0
)

print("\nSection 3 result:", "PASS" if section_3_pass else "CHECK")

Action playbook validation
--------------------------
Required columns present: True
Rank sequence valid: True
Score range valid: True
Reason codes valid: True
Actions valid: True
Duplicate client-content pairs: 0

Section 3 result: PASS


## 4. Monitoring / retrain triggers

### Evidence and Claim Check

The action playbook is based on observed historical performance signals from the March 2026 content-performance data.

The ranked scores are intended to support prioritization and human review. They should be interpreted as directional decision-support evidence rather than causal conclusions.

A high score indicates that the observed performance signals meet the defined baseline scoring conditions. It does not guarantee that the content will decline or that the recommended review action will improve future performance.

Human review is required before taking action.

In [7]:
# ML-10 — Section 4: Evidence and claim check

claim_text = """
The action playbook uses observed historical performance signals
to prioritize content for human review. The scores provide
directional decision-support evidence and are not causal claims
or guarantees of future performance.
"""

print("========== SECTION 4 CLAIM CHECK ==========")

print("\nEvidence basis:")
print("PASS - Uses observed historical performance signals.")

print("\nDecision-support:")
print("PASS - Used for prioritization and human review.")

print("\nCausal claim:")
print("PASS - No causal claim is made.")

print("\nGuarantee:")
print("PASS - No guarantee of future performance is made.")

print("\nHuman review:")
print("PASS - Human review is required before action.")

print("\nClaim statement:")
print(claim_text.strip())

print("\nSection 4 result: PASS")

========== SECTION 4 CLAIM CHECK ==========

Evidence basis:
PASS - Uses observed historical performance signals.

Decision-support:
PASS - Used for prioritization and human review.

Causal claim:
PASS - No causal claim is made.

Guarantee:
PASS - No guarantee of future performance is made.

Human review:
PASS - Human review is required before action.

Claim statement:
The action playbook uses observed historical performance signals
to prioritize content for human review. The scores provide
directional decision-support evidence and are not causal claims
or guarantees of future performance.

Section 4 result: PASS


## 5. Exports for the paper

### Final Output Check

The ML-10 action playbook produces a ranked content-review queue with reason codes and recommended actions.

The final output contains 176,738 content-client records with the required fields and no duplicate client-content pairs.

The playbook is intended for human review and decision support. The recommendations are directional and should not be interpreted as causal conclusions or guaranteed outcomes.

**Final result: PASS**

In [8]:
# ML-10 — Section 5: Final output check

print("========== ML-10 FINAL CHECK ==========")

print("Total action items:", len(ranked_action_queue))

print(
    "Required columns present:",
    all(
        col in ranked_action_queue.columns
        for col in [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "reason_code",
            "action"
        ]
    )
)

print(
    "Duplicate client-content pairs:",
    ranked_action_queue[
        ["client_hash_id", "content_hash_id"]
    ].duplicated().sum()
)

print(
    "Valid score range:",
    ranked_action_queue["score"].between(0, 5).all()
)

print(
    "Reason code valid:",
    ranked_action_queue["reason_code"]
    .eq("LOW_SEARCH_PERFORMANCE")
    .all()
)

print(
    "Action valid:",
    ranked_action_queue["action"]
    .eq("REVIEW_REFRESH")
    .all()
)

final_pass = (
    len(ranked_action_queue) == 176738
    and all(
        col in ranked_action_queue.columns
        for col in [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "reason_code",
            "action"
        ]
    )
    and ranked_action_queue[
        ["client_hash_id", "content_hash_id"]
    ].duplicated().sum() == 0
    and ranked_action_queue["score"].between(0, 5).all()
    and ranked_action_queue["reason_code"]
    .eq("LOW_SEARCH_PERFORMANCE")
    .all()
    and ranked_action_queue["action"]
    .eq("REVIEW_REFRESH")
    .all()
)

print("\nML-10 FINAL RESULT:", "PASS" if final_pass else "CHECK")

========== ML-10 FINAL CHECK ==========
Total action items: 176738
Required columns present: True
Duplicate client-content pairs: 0
Valid score range: True
Reason code valid: True
Action valid: True

ML-10 FINAL RESULT: PASS


## Self-check

Before you submit, confirm each line honestly:

* [x] Every section above is filled — markdown thinking AND the code that backs it
* [x] The notebook runs top to bottom with no errors (Runtime → Run all)
* [x] No client names, URLs, or private queries anywhere
* [x] My claims use careful words: observed, measured, directional, decision-support
* [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
